# Setup

In [2]:
!pip install -q -U trl bitsandbytes transformers accelerate javalang python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 11.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.9 MB/s eta 0:00:00:00:0100:01


In [1]:
!pip install -q tqdm

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch
from tqdm import tqdm
import subprocess
import os
from pathlib import Path

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

def get_hf_token():
    t = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if t:
        return str(t).strip() or None
    try:
        from google.colab import userdata
        u = userdata.get("HF_TOKEN")
        if u:
            return str(u).strip() or None
    except Exception:
        pass
    return None

def get_github_token():
    t = os.getenv("GITHUB_TOKEN") or os.getenv("GH_TOKEN")
    if t:
        return str(t).strip() or None
    try:
        from google.colab import userdata
        u = userdata.get("GITHUB_TOKEN")
        if u:
            return str(u).strip() or None
    except Exception:
        pass
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ.setdefault("HF_TOKEN", HF_TOKEN)
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)


In [8]:
from huggingface_hub import login, notebook_login
from trl import SFTTrainer, SFTConfig

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    notebook_login()


In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # top of notebook

## Global Variables

In [ ]:
# Ensure the dataset name matches your target on Hugging Face
hf_dataset_name = "marcuswu5/fabric-code-dataset-v2"

#Model used for labeling data with instructions
LABEL_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

#The base model that is being fine tuned
BASE_MODEL = "deepseek-ai/deepseek-coder-6.7b-instruct"

#To disconnect immediately after training and uploading
AUTO_DISCONNECT = True
successful_upload = False #prevents autodisconnect when something fails

MAX_LABEL_TOKENS = 2048


MinHash Near-Deduplication
==========================
A self-contained implementation of MinHash + LSH near-deduplication,
mirroring the approach used by the BigCode project for The Stack dataset.

Pipeline
--------
1. Shingling   – tokenise each document into a set of character n-grams (shingles).
2. MinHashing  – compress each shingle set into a fixed-length signature using
                 `num_perm` random hash functions (universal hashing via the
                 Mersenne-prime trick, same as bigcode-dataset).
3. LSH banding – split each signature into `B` bands of `R` rows; documents
                 that share an identical band are candidate duplicates.
4. Union-Find  – group candidate pairs into connected components (clusters).
5. Filtering   – keep one representative per cluster; discard the rest.

Dependencies
------------
    pip install numpy

References
----------
- https://github.com/bigcode-project/bigcode-dataset/tree/main/near_deduplication
- https://huggingface.co/blog/dedup

## Helpers

In [ ]:
from __future__ import annotations

import hashlib
import re
import struct
from dataclasses import dataclass, field
from typing import Callable, Iterable, List, Optional, Tuple

import numpy as np


# ---------------------------------------------------------------------------
# Constants (matching bigcode-dataset)
# ---------------------------------------------------------------------------

MERSENNE_PRIME: int = (1 << 61) - 1   # largest Mersenne prime that fits uint64
MAX_HASH: int = (1 << 32) - 1          # 32-bit hash cap (keeps values manageable)
_RNG = np.random.RandomState(42)


# ---------------------------------------------------------------------------
# Union-Find (Disjoint-Set)
# ---------------------------------------------------------------------------

class UnionFind:
    """Weighted quick-union with path compression."""

    def __init__(self) -> None:
        self._parent: dict[int, int] = {}

    def find(self, x: int) -> int:
        if x not in self._parent:
            self._parent[x] = x
        while self._parent[x] != x:
            self._parent[x] = self._parent[self._parent[x]]  # path halving
            x = self._parent[x]
        return x

    def union(self, x: int, y: int) -> None:
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self._parent[rx] = ry


# ---------------------------------------------------------------------------
# Optimal band / row computation
# ---------------------------------------------------------------------------

def optimal_bands_and_rows(threshold: float, num_perm: int) -> Tuple[int, int]:
    """
    Find the (B, R) pair that maximises sensitivity at `threshold` for a
    signature of length `num_perm`, i.e. minimises |threshold - (1/B)^(1/R)|.
    """
    best_error = float("inf")
    best_b, best_r = 1, num_perm
    for b in range(1, num_perm + 1):
        if num_perm % b != 0:
            continue
        r = num_perm // b
        t = (1 / b) ** (1 / r)
        err = abs(t - threshold)
        if err < best_error:
            best_error, best_b, best_r = err, b, r
    return best_b, best_r


# ---------------------------------------------------------------------------
# Shingling
# ---------------------------------------------------------------------------

_TOKEN_PATTERN = re.compile(r"\w+")

def ngrams(text: str, n: int) -> set[str]:
    """Return the set of whitespace-token n-grams from *text*."""
    tokens = _TOKEN_PATTERN.findall(text.lower())
    if len(tokens) < n:
        return set()
    return {" ".join(tokens[i : i + n]) for i in range(len(tokens) - n + 1)}


def _hash_shingle(shingle: str) -> int:
    """Stable 32-bit hash of a shingle string (SHA-1 prefix)."""
    return struct.unpack("<I", hashlib.sha1(shingle.encode("utf-8")).digest()[:4])[0]


# ---------------------------------------------------------------------------
# MinHash signature
# ---------------------------------------------------------------------------

def _build_permutations(num_perm: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Sample `num_perm` (a, b) pairs for universal hashing:
        h_i(x) = (a_i * x + b_i) mod MERSENNE_PRIME
    """
    a = _RNG.randint(1, MERSENNE_PRIME, size=num_perm, dtype=np.uint64)
    b = _RNG.randint(0, MERSENNE_PRIME, size=num_perm, dtype=np.uint64)
    return a, b


def compute_minhash(
    shingles: set[str],
    a: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    """
    Compute a MinHash signature vector of length `num_perm`.

    Parameters
    ----------
    shingles : set of shingle strings
    a, b     : permutation coefficients from `_build_permutations`

    Returns
    -------
    np.ndarray of shape (num_perm,), dtype uint64
        Contains MAX_HASH for empty documents (no shingles).
    """
    num_perm = len(a)
    sig = np.full(num_perm, MAX_HASH, dtype=np.uint64)

    if not shingles:
        return sig

    hashes = np.array([_hash_shingle(s) for s in shingles], dtype=np.uint64)

    # Vectorised universal hash for every (permutation × shingle) pair
    # shape: (num_perm, |shingles|)
    phashes = (
        (a[:, None] * hashes[None, :] + b[:, None]) % MERSENNE_PRIME
    ) & MAX_HASH

    np.minimum(sig, phashes.min(axis=1), out=sig)
    return sig


# ---------------------------------------------------------------------------
# LSH index
# ---------------------------------------------------------------------------

class LSHIndex:
    """
    Banded LSH index.

    Documents are added one by one; `candidates()` returns all (i, j) pairs
    that share at least one band bucket.
    """

    def __init__(self, B: int, R: int) -> None:
        self.B = B
        self.R = R
        # band_id → {bucket_key → [doc_ids]}
        self._buckets: list[dict[bytes, list[int]]] = [{} for _ in range(B)]

    def add(self, doc_id: int, sig: np.ndarray) -> None:
        for band_idx in range(self.B):
            band = sig[band_idx * self.R : (band_idx + 1) * self.R]
            key = band.tobytes()
            self._buckets[band_idx].setdefault(key, []).append(doc_id)

    def candidates(self) -> Iterable[Tuple[int, int]]:
        seen: set[Tuple[int, int]] = set()
        for band_buckets in self._buckets:
            for ids in band_buckets.values():
                if len(ids) < 2:
                    continue
                for i in range(len(ids)):
                    for j in range(i + 1, len(ids)):
                        pair = (ids[i], ids[j])
                        if pair not in seen:
                            seen.add(pair)
                            yield pair

## Public API

In [ ]:

@dataclass
class DeduplicationResult:
    """Return value of :func:`minhash_dedup`."""

    deduplicated: List[str]
    """Texts that survived deduplication (one per cluster)."""

    duplicate_indices: List[int]
    """Original indices of texts that were removed."""

    cluster_map: dict = field(default_factory=dict)
    """Maps each original index → representative index in its cluster."""


def minhash_dedup(
    texts: List[str],
    *,
    ngram_size: int = 5,
    num_perm: int = 256,
    threshold: float = 0.7,
    min_ngram_count: int = 5,
    bands: Optional[int] = None,
    rows: Optional[int] = None,
) -> DeduplicationResult:
    """
    Near-deduplicate a list of texts using MinHash + LSH.

    Parameters
    ----------
    texts : list of str
        Input documents.
    ngram_size : int
        Token n-gram size for shingling (bigcode default: 5).
    num_perm : int
        Number of MinHash permutations / signature length (bigcode default: 256).
    threshold : float
        Jaccard similarity threshold above which two docs are considered
        duplicates (bigcode default: 0.7).
    min_ngram_count : int
        Documents with fewer n-grams than this are treated as too short and
        removed outright (bigcode default: 5).
    bands : int, optional
        Number of LSH bands (auto-computed from threshold if None).
    rows : int, optional
        Number of rows per band (auto-computed from threshold if None).

    Returns
    -------
    DeduplicationResult
        See :class:`DeduplicationResult` for fields.

    Example
    -------
    >>> texts = [
    ...     "The quick brown fox jumps over the lazy dog",
    ...     "The quick brown fox jumps over the lazy dog!",   # near-duplicate
    ...     "A completely different sentence about cats.",
    ... ]
    >>> result = minhash_dedup(texts, threshold=0.7)
    >>> result.deduplicated
    ['The quick brown fox jumps over the lazy dog', 'A completely different sentence about cats.']
    >>> result.duplicate_indices
    [1]
    """
    if not texts:
        return DeduplicationResult(
            deduplicated=[], duplicate_indices=[], cluster_map={}
        )

    # --- Band / row parameters -------------------------------------------
    if bands is None or rows is None:
        B, R = optimal_bands_and_rows(threshold, num_perm)
    else:
        B, R = bands, rows

    # --- Build permutation coefficients -----------------------------------
    a, b = _build_permutations(num_perm)

    # --- Compute signatures & index ---------------------------------------
    lsh = LSHIndex(B, R)
    signatures: list[Optional[np.ndarray]] = []
    too_short: set[int] = set()

    for idx, text in enumerate(texts):
        shingles = ngrams(text, ngram_size)
        if len(shingles) < min_ngram_count:
            too_short.add(idx)
            signatures.append(None)
            continue
        sig = compute_minhash(shingles, a, b)
        signatures.append(sig)
        lsh.add(idx, sig)

    # --- Cluster via Union-Find ------------------------------------------
    uf = UnionFind()

    for i, j in lsh.candidates():
        # Both must have valid signatures (not too-short docs)
        if signatures[i] is None or signatures[j] is None:
            continue
        uf.union(i, j)

    # --- Build cluster map & choose representatives ----------------------
    # Representative of a cluster = the document with the smallest index.
    representative: dict[int, int] = {}  # root → representative index

    for idx in range(len(texts)):
        if idx in too_short:
            continue
        root = uf.find(idx)
        if root not in representative:
            representative[root] = idx

    cluster_map: dict[int, int] = {}
    duplicate_indices: list[int] = []
    keep: set[int] = set()

    for idx in range(len(texts)):
        if idx in too_short:
            duplicate_indices.append(idx)
            cluster_map[idx] = idx  # maps to itself; no representative
            continue
        root = uf.find(idx)
        rep = representative[root]
        cluster_map[idx] = rep
        if idx == rep:
            keep.add(idx)
        else:
            duplicate_indices.append(idx)

    deduplicated = [texts[i] for i in sorted(keep)]

    return DeduplicationResult(
        deduplicated=deduplicated,
        duplicate_indices=sorted(duplicate_indices),
        cluster_map=cluster_map,
    )

#Obtain Dataset

## Search for fabric repos

In [ ]:
import requests
import time

github_token = get_github_token()
if not github_token:
    raise ValueError(
        "GITHUB_TOKEN not set. Add GITHUB_TOKEN to your .env or Colab Secrets (or set GH_TOKEN)."
    )

headers = {
    "Authorization": f"token {github_token}",
    "Accept": "application/vnd.github.v3+json"
}

def search_fabric_repos(page=1):
    params = {
        "q": "topic:fabric topic:minecraft language:Java pushed:>2025-01-01",
        "order": "desc",
        "per_page": 100,
        "page": page
    }
    r = requests.get("https://api.github.com/search/repositories",
                     headers=headers, params=params)
    return r.json()

repos = []
for page in tqdm(range(1, 11)):  # up to 1000 results
    data = search_fabric_repos(page)
    repos.extend(data["items"])
    time.sleep(2)  # stay under rate limit

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
print(repos[0]["topics"])

['azdo', 'fabric', 'minecraft', 'neoforge', 'opengl']


In [ ]:
for repo in repos:
  if "fabric" in repo["full_name"].lower():
    print(repo["full_name"])

NameError: name 'repos' is not defined

In [ ]:
def is_worth_cloning(repo):
    if repo["stargazers_count"] < 3:
        return False
    if repo["size"] < 50:        # kb — too small to be a real mod
        return False
    if repo["size"] > 500000:    # kb — probably a monorepo or game asset dump
        return False
    if repo["fork"] and repo["stargazers_count"] < 10:
        return False             # forks of tutorial repos add noise
    if "fabricmc" in repo["full_name"].lower():
        return False             # Remove the actual fabric mods
    if repo["license"] and repo["license"]["spdx_id"]  not in {"MIT", "Apache-2.0", "BSD-2-Clause", "BSD-3-Clause", "CC0-1.0", "CC-BY-4.0", "NOASSERTION"}:
        return False
    if repo["topics"] and "neoforge" in repo["topics"]:
        return False
    return True

def clone_repo(repo, dest_dir="./repos"):
    name = repo["full_name"].replace("/", "__")
    path = Path(dest_dir) / name

    if path.exists():
        return {"status": "skipped", "repo": repo, "path": path}

    result = subprocess.run([
        "git", "clone",
        "--depth", "1",
        "--single-branch",
        "--filter=blob:limit=1m",
        repo["clone_url"],
        str(path)
    ], capture_output=True, timeout=60)

    if result.returncode != 0:
        return {"status": "failed", "repo": repo, "error": result.stderr.decode()}

    return {"status": "cloned", "repo": repo, "path": path}

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import subprocess

def clone_all(repos, dest_dir="./repos", max_workers=12):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)

    results = {"cloned": [], "skipped": [], "failed": []}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(clone_repo, repo, dest_dir): repo for repo in repos}

        with tqdm(as_completed(futures), total=len(futures), unit="repo") as pbar:
          for future in pbar:
              result = future.result()
              status = result["status"]

              if status == "cloned":
                  results["cloned"].append(result["repo"])
              elif status == "skipped":
                  results["skipped"].append(result["repo"])
              else:
                  results["failed"].append(result)

              pbar.set_postfix({
                  "cloned": len(results["cloned"]),
                  "skipped": len(results["skipped"]),
                  "failed": len(results["failed"])
              })

    print(f"\nDone — {len(results['cloned'])} cloned, "
          f"{len(results['skipped'])} skipped, "
          f"{len(results['failed'])} failed")
    return results

In [ ]:
good_repos = [r for r in tqdm(repos) if is_worth_cloning(r)]
print("\n%d repos remaining after filtering from %d repos" % (len(good_repos), len(repos)))


100%|██████████| 1000/1000 [00:00<00:00, 722159.78it/s]


202 repos remaining after filtering from 1000 repos


In [ ]:
results = clone_all(good_repos, dest_dir="./repos", max_workers=12)

100%|██████████| 202/202 [00:31<00:00,  6.47repo/s, cloned=202, skipped=0, failed=0]


Done — 202 cloned, 0 skipped, 0 failed


## Save Raw Data as Dataset

In [ ]:
from pathlib import Path

def extract_java_files(repo_path):
    java_files = []
    for f in Path(repo_path).rglob("*.java"):
        # skip generated, test, and build output
        parts = f.parts
        if any(p in parts for p in ["build", "test", "generated", ".gradle"]):
            continue
        if f.stat().st_size < 200:    # skip stub files
            continue
        if f.stat().st_size > 80000:  # skip huge generated files
            continue
        java_files.append(f)
    return java_files

def classify_file(source):
    """Classify by what kind of Fabric code this is."""
    if "@Mixin" in source:
        return "mixin"
    if "implements ModInitializer" in source:
        return "entrypoint"
    if "Registry.register" in source or "RegistryKey" in source:
        return "registry"
    if "ServerPlayerEntity" in source or "PlayerEntity" in source:
        return "player_logic"
    if "BlockItem" in source or "extends Block" in source:
        return "block_item"
    if "FabricDataGenHelper" in source or "FabricDataOutput" in source:
        return "datagen"
    return "general"


In [ ]:
import re

def remove_brackets_and_commas(input_string):
    """Removes all brackets, parentheses, and commas from a string."""
    # Remove square brackets []
    s = re.sub(r'\[.*?\]', '', input_string)
    # Remove parentheses ()
    s = re.sub(r'\(.*?\)', '', s)
    # Remove commas
    s = s.replace(',', '')
    return s.strip()

In [ ]:
import hashlib
from pathlib import Path
import re

def extract_version_from_gradle(gradle_path_str):
    """Pull the Minecraft and Fabric versions from gradle files."""
    if not gradle_path_str:
        return None, None

    gradle_path = Path(gradle_path_str)
    repo_dir = gradle_path.parent

    mc_version = None
    fabric_version = None

    # 1. Check gradle.properties
    if gradle_path.exists():
        content = gradle_path.read_text(errors='ignore')
        for line in content.splitlines():
            line = line.strip()
            if line.startswith("minecraft_version"):
                mc_version = line.split("=")[1].strip()
            elif any(line.startswith(k) for k in ["fabric_version", "fabric_loader_version", "fabricVersion"]):
                fabric_version = line.split("=")[1].strip()

    # 2. Check gradle/libs.versions.toml if needed
    if not mc_version or not fabric_version:
        toml_path = repo_dir / "gradle" / "libs.versions.toml"
        if toml_path.exists():
            content = toml_path.read_text(errors='ignore')
            in_versions_section = False
            for line in content.splitlines():
                line = line.strip()
                if line == "[versions]":
                    in_versions_section = True
                    continue
                elif line.startswith("["):
                    in_versions_section = False

                if in_versions_section and "=" in line:
                    key, val = [p.strip() for p in line.split("=", 1)]
                    val = val.strip('"\'')
                    if key in ["minecraft", "minecraft_version", "minecraftVersion"] and not mc_version:
                        mc_version = val
                    elif key in ["fabric", "fabric_version", "fabric_loader", "fabricLoader", "fabric_api", "fabricApi"] and not fabric_version:
                        fabric_version = val

    # 3. Check build.gradle or build.gradle.kts if needed
    if not mc_version or not fabric_version:
        for build_file in [repo_dir / "build.gradle", repo_dir / "build.gradle.kts"]:
            if build_file.exists():
                content = build_file.read_text(errors='ignore')
                # Simple regex to catch basic string declarations or dependencies
                if not mc_version:
                    mc_match = re.search(r'minecraft\s+["\'](.*?)["\']', content) or re.search(r'minecraft\s*\(["\'](?:com\.mojang:)?minecraft:(.*?)["\']\)', content)
                    if mc_match:
                        mc_version = mc_match.group(1)
                if not fabric_version:
                    fab_match = re.search(r'fabric-api:(.*?)["\']', content) or re.search(r'fabric_loader\s+["\'](.*?)["\']', content)
                    if fab_match:
                        fabric_version = fab_match.group(1)

    if mc_version and not any(c.isdigit() for c in mc_version):
        mc_version = None
    if fabric_version and not any(c.isdigit() for c in fabric_version):
        fabric_version = None

    mc_version = remove_brackets_and_commas(mc_version)
    fabric_version = remove_brackets_and_commas(fabric_version)

    return mc_version, fabric_version

def make_training_example_separated(description, code, metadata):
    # detect and include fabric version if findable
    mc_version, fabric_version = extract_version_from_gradle(metadata.get("gradle_path"))

    if mc_version and fabric_version:
        context = f"Minecraft {mc_version}, Fabric {fabric_version}"
    elif mc_version:
        context = f"Minecraft {mc_version}, Fabric mod"
    elif fabric_version:
        context = f"Fabric {fabric_version}"
    else:
        context = "Fabric mod"

    return {
        "instruction": description,
        "context": context,
        "code": code,
        "metadata": {
            "source_repo": metadata["repo"],
            "file_type": metadata["type"],
            "minecraft_version": mc_version,
            "fabric_version": fabric_version,
            "sha": hashlib.md5(code.encode()).hexdigest()
        }
    }


In [ ]:
import hashlib
from pathlib import Path
from tqdm.auto import tqdm

# Assuming `make_training_example_separated`, `classify_file`, `extract_java_files`,
# and `extract_version_from_gradle` are defined in previous cells.

raw_separated_examples = []

repo_dirs = [d for d in Path("./repos").iterdir() if d.is_dir()]

for repo_path in tqdm(repo_dirs, desc="Processing Repos"):
    gradle_props = repo_path / "gradle.properties"
    java_files = extract_java_files(repo_path)

    for jf in java_files:
        try:
            source_code = jf.read_text()
            file_type = classify_file(source_code) # Fixed: passed only one argument

            metadata = {
                "repo": repo_path.name,
                "type": file_type,
                "gradle_path": str(gradle_props) if gradle_props.exists() else None
            }

            # Create the separated example with None for the description
            example = make_training_example_separated(None, source_code, metadata)
            raw_separated_examples.append(example)
        except Exception as e:
            # Skip files that can't be read or processed
            continue

print(f"\nExtracted {len(raw_separated_examples)} raw separated examples.")


Processing Repos:   0%|          | 0/202 [00:00<?, ?it/s]

NameError: name 'extract_java_files' is not defined

# Load Existing Dataset

In [ ]:
import pandas as pd

def print_dataset_statistics(dataset):
    if not dataset:
        print("Dataset is empty.")
        return

    df = pd.DataFrame(dataset)

    # Extract metadata into separate columns for easier analysis
    df['file_type'] = df['metadata'].apply(lambda x: x.get('file_type', 'unknown'))
    df['fabric_version'] = df['metadata'].apply(lambda x: x.get('fabric_version', 'unknown'))
    df['minecraft_version'] = df['metadata'].apply(lambda x: x.get('minecraft_version', 'unknown'))

    # Handle length based on whether we have 'code' or 'text'
    if 'code' in df.columns:
        df['code_length'] = df['code'].str.len()
    elif 'text' in df.columns:
        df['code_length'] = df['text'].str.len()

    print("--- Dataset Summary ---")
    print("\nCategory Distribution (File Type):")
    print(df['file_type'].value_counts(dropna=False))

    print("\nFabric Version Distribution:")
    print(df['fabric_version'].value_counts(dropna=False))

    print("\nMinecraft Version Distribution:")
    print(df['minecraft_version'].value_counts(dropna=False))

    if 'code_length' in df.columns:
        print("\nCode/Text Length Distribution (Characters):")
        print(df['code_length'].describe())

    # Count examples where 'instruction' is None
    if 'instruction' in df.columns:
        none_instruction_count = df['instruction'].isna().sum()
    else:
        none_instruction_count = len(df)

    print(f"\nNumber of examples with 'instruction' as None: {none_instruction_count}")

# Call the function on the raw dataset
if 'raw_separated_examples' in globals():
    print("Statistics for raw_separated_examples:")
    print_dataset_statistics(raw_separated_examples)
else:
    print("raw_separated_examples not found.")


raw_separated_examples not found.


In [ ]:
from datasets import load_dataset
import hashlib

try:
    # Load the existing dataset from Hugging Face
    print("Loading existing dataset from Hugging Face...")
    loaded_ds = load_dataset(hf_dataset_name, split="train", token=HF_TOKEN)

    # Convert back to list of dictionaries
    all_examples = loaded_ds.to_list()

    # Re-build the existing_shas set for the processing loop
    existing_shas = {ex['metadata']['sha'] for ex in all_examples}

    print(f"Successfully loaded {len(all_examples)} existing examples.")
    print(f"Deduplication set contains {len(existing_shas)} unique SHAs.")
except Exception as e:
    print(f"Could not load dataset (it might not exist yet): {e}")
    if 'all_examples' not in globals():
        all_examples = []
    if 'existing_shas' not in globals():
        existing_shas = set()

Loading existing dataset from Hugging Face...


README.md:   0%|          | 0.00/606 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6450 [00:00<?, ? examples/s]

Successfully loaded 6450 existing examples.
Deduplication set contains 6450 unique SHAs.


In [ ]:
combined_examples_dict = {}

# First add raw separated examples (lower priority)
if 'raw_separated_examples' in globals():
    for ex in raw_separated_examples:
        sha = ex['metadata']['sha']
        combined_examples_dict[sha] = ex

# Then add loaded examples (higher priority, will overwrite duplicates)
if 'all_examples' in globals():
    for ex in all_examples:
        sha = ex['metadata']['sha']
        combined_examples_dict[sha] = ex

# Convert back to list
combined_dataset = list(combined_examples_dict.values())

print(f"Loaded dataset size: {len(all_examples) if 'all_examples' in globals() else 0}")
print(f"Raw dataset size: {len(raw_separated_examples) if 'raw_separated_examples' in globals() else 0}")
print(f"Total combined unique examples: {len(combined_dataset)}")


Loaded dataset size: 6450
Raw dataset size: 0
Total combined unique examples: 6450


In [ ]:
if 'combined_dataset' in globals() and len(combined_dataset) > 0:
    print("Statistics for combined_dataset:")
    print_dataset_statistics(combined_dataset)
else:
    print("No combined examples found. Please ensure the extraction and combination cells ran successfully.")

Statistics for combined_dataset:
--- Dataset Summary ---

Category Distribution (File Type):
file_type
general         4562
mixin           1170
player_logic     308
registry         220
block_item        81
entrypoint        72
datagen           37
Name: count, dtype: int64

Fabric Version Distribution:
fabric_version
0.16.0             694
0.139.4+1.21.11    662
None               580
0.18.4             498
0.114.0+1.21.4     311
                  ... 
0.138.0+1.21.10      5
0.134.0+1.21.9       4
0.97.2+1.20.4        4
0.121.0+1.21.5       4
0.102.0+1.21.1       3
Name: count, Length: 67, dtype: int64

Minecraft Version Distribution:
minecraft_version
1.21.11                                    1255
1.21                                        753
None                                        580
1.21.4                                      562
1.20.1                                      426
26.1                                        382
1.18.2                                      369
b

## Run Minhash Dedup

In [ ]:
import time

if 'combined_dataset' in globals():
    print(f"Starting deduplication of {len(combined_dataset)} examples...")
    start_time = time.time()

    # Extract the code blocks to compare
    texts_to_dedup = [ex['code'] for ex in combined_dataset]

    # Run minhash near-deduplication
    dedup_result = minhash_dedup(
        texts=texts_to_dedup,
        threshold=0.7 # Default BigCode threshold
    )

    # Filter out the duplicates using their original indices
    duplicate_indices_set = set(dedup_result.duplicate_indices)
    deduped_dataset = [ex for i, ex in enumerate(combined_dataset) if i not in duplicate_indices_set]

    print(f"Deduplication finished in {time.time() - start_time:.2f} seconds.")
    print(f"Original size: {len(combined_dataset)}")
    print(f"Duplicates removed: {len(dedup_result.duplicate_indices)}")
    print(f"New dataset size: {len(deduped_dataset)}")

    # Update the main dataset variable
    combined_dataset = deduped_dataset
else:
    print("combined_dataset is not defined. Please run the data extraction and combination cells first.")

Starting deduplication of 6450 examples...


NameError: name 'minhash_dedup' is not defined

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer for {LABEL_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(LABEL_MODEL, trust_remote_code=True)


Loading tokenizer for Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
if 'combined_dataset' in globals() and 'tokenizer' in globals():
    initial_size = len(combined_dataset)
    print(f"Filtering dataset (Limit: %d tokens)..." % (MAX_LABEL_TOKENS - 45))

    # Filter the list by encoding each snippet and checking length
    combined_dataset = [
        ex for ex in tqdm(combined_dataset, desc="Filtering")
        if len(tokenizer.encode(ex.get('code', ''))) <= (MAX_LABEL_TOKENS - 45)
    ]

    removed = initial_size - len(combined_dataset)
    print(f"\nFiltering complete.")
    print(f"Removed {removed} examples exceeding %d tokens." % (MAX_LABEL_TOKENS - 45))
    print(f"New dataset size: {len(combined_dataset)}")
else:
    print("Dataset or tokenizer not found. Please ensure previous cells have run.")

Filtering dataset (Limit: 2003 tokens)...


Filtering: 100%|██████████| 6450/6450 [00:09<00:00, 670.81it/s]


Filtering complete.
Removed 0 examples exceeding 2003 tokens.
New dataset size: 6450


## Use tokenizer to view distribution of sizes

In [ ]:

# if 'combined_dataset' in globals() and len(combined_dataset) > 0:
#     num_samples = 5
#     print(f"\nToken counts for the first {num_samples} code examples:")
#     for i, ex in enumerate(combined_dataset[:num_samples]):
#         code_text = ex.get('code', '')
#         token_ids = tokenizer.encode(code_text)
#         print(f"Example {i+1}: {len(token_ids)} tokens (character length: {len(code_text)})")
# else:
#     print("combined_dataset is not defined or is empty.")

Loading tokenizer for Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Token counts for the first 5 code examples:
Example 1: 293 tokens (character length: 1434)
Example 2: 71 tokens (character length: 377)
Example 3: 314 tokens (character length: 1409)
Example 4: 1058 tokens (character length: 4024)
Example 5: 271 tokens (character length: 1201)


In [ ]:
# from tqdm.auto import tqdm

# if 'combined_dataset' in globals() and 'tokenizer' in globals() and len(combined_dataset) > 0:
#     max_tokens = 0
#     max_index = -1

#     print(f"Calculating maximum tokens across {len(combined_dataset)} examples...")
#     for i, ex in enumerate(tqdm(combined_dataset, desc="Tokenizing")):
#         code_text = ex.get('code', '')
#         # encode without adding special tokens to get pure code token count if desired,
#         # but standard encode is fine for general max length checking.
#         token_count = len(tokenizer.encode(code_text))
#         if token_count > max_tokens:
#             max_tokens = token_count
#             max_index = i

#     print(f"\nThe maximum number of tokens in a single code example is: {max_tokens}")
#     print(f"This occurs at dataset index: {max_index}")
# else:
#     print("combined_dataset or tokenizer is not defined. Please run the previous cells.")

Calculating maximum tokens across 7009 examples...


Tokenizing:   0%|          | 0/7009 [00:00<?, ?it/s]


The maximum number of tokens in a single code example is: 24184
This occurs at dataset index: 5524


In [ ]:
# import pandas as pd
# import numpy as np

# if 'combined_dataset' in globals() and 'tokenizer' in globals():
#     print("Tokenizing dataset to calculate distribution...")
#     token_counts = [len(tokenizer.encode(ex.get('code', ''))) for ex in tqdm(combined_dataset)]

#     # Create bins of 1000
#     max_val = max(token_counts)
#     bins = np.arange(0, max_val + 1000, 1000)
#     labels = [f"{i}-{i+999}" for i in bins[:-1]]

#     # Create dataframe for analysis
#     df_tokens = pd.DataFrame(token_counts, columns=['token_count'])
#     df_tokens['range'] = pd.cut(df_tokens['token_count'], bins=bins, labels=labels, right=False)

#     # Display counts
#     distribution = df_tokens['range'].value_counts().sort_index().reset_index()
#     distribution.columns = ['Token Range', 'Count']

#     display(distribution)
# else:
#     print("Dataset or tokenizer not found.")

Tokenizing dataset to calculate distribution...


  0%|          | 0/7009 [00:00<?, ?it/s]

,Token Range,Count
0,0-999,5443
1,1000-1999,1000
2,2000-2999,296
3,3000-3999,110
4,4000-4999,62
5,5000-5999,39
6,6000-6999,19
7,7000-7999,11
8,8000-8999,8
9,9000-9999,3


In [ ]:
# if 'combined_dataset' in globals() and 'tokenizer' in globals():
#     initial_size = len(combined_dataset)
#     print(f"Filtering dataset (Limit: 4000 tokens)...")

#     # Filter the list by encoding each snippet and checking length
#     combined_dataset = [
#         ex for ex in tqdm(combined_dataset, desc="Filtering")
#         if len(tokenizer.encode(ex.get('code', ''))) <= 4000
#     ]

#     removed = initial_size - len(combined_dataset)
#     print(f"\nFiltering complete.")
#     print(f"Removed {removed} examples exceeding 4000 tokens.")
#     print(f"New dataset size: {len(combined_dataset)}")
# else:
#     print("Dataset or tokenizer not found. Please ensure previous cells have run.")

Filtering dataset (Limit: 4000 tokens)...


Filtering:   0%|          | 0/6849 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Upload the combined dataset

In [ ]:
from datasets import Dataset
import pandas as pd
import datetime


if 'combined_dataset' in globals() and len(combined_dataset) > 0:
    print(f"Preparing to upload {len(combined_dataset)} examples from combined_dataset...")
    df_combined = pd.DataFrame(combined_dataset)
    combined_hf_dataset = Dataset.from_pandas(df_combined)

    if "__index_level_0__" in combined_hf_dataset.column_names:
        combined_hf_dataset = combined_hf_dataset.remove_columns(["__index_level_0__"])

    combined_hf_dataset = combined_hf_dataset.shuffle(seed=42)

    try:
        token = get_hf_token()
        if not token:
            print("HF_TOKEN not found. Set HF_TOKEN in .env or Colab Secrets.")
        else:
            # Generate a commit message with current time and date
            current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            commit_message = f"Update dataset on {current_time}"
            combined_hf_dataset.push_to_hub(hf_dataset_name, token=token, private=True, commit_message=commit_message)
            print(f"Successfully pushed combined dataset to: https://huggingface.co/datasets/{hf_dataset_name}")
    except Exception as e:
        print(f"Error pushing to Hub: {e}")
else:
    print("combined_dataset is not defined or is empty.")

Preparing to upload 6450 examples from combined_dataset...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   9%|8         |  529kB / 6.21MB            

Successfully pushed combined dataset to: https://huggingface.co/datasets/marcuswu5/fabric-code-dataset-v2


## Separate the labeled and unlabeled data

In [ ]:
def separate_none_instructions(dataset):
    none_data = []
    valid_data = []

    for item in dataset:
        # Check if instruction is None or key is missing
        if item.get('instruction') is None:
            none_data.append(item)
        else:
            valid_data.append(item)

    return none_data, valid_data

if 'combined_dataset' in globals():
    none_instructions, valid_instructions = separate_none_instructions(combined_dataset)

    print(f"Data with 'None' instructions: {len(none_instructions)}")
    print(f"Data with valid instructions: {len(valid_instructions)}")
else:
    print("combined_dataset not found.")


Data with 'None' instructions: 6205
Data with valid instructions: 245


# Label each piece of data using an LLM

In [ ]:
if 'none_instructions' in globals() and 'valid_instructions' in globals():
    valid_shas = {item['metadata']['sha'] for item in valid_instructions}
    initial_none_count = len(none_instructions)

    # Filter out items from none_instructions that are already in valid_instructions
    # (identified by their SHA)
    none_instructions = [item for item in none_instructions if item['metadata']['sha'] not in valid_shas]

    deduplicated_count = initial_none_count - len(none_instructions)
    print(f"Removed {deduplicated_count} items from `none_instructions` that were already processed.")
    print(f"Remaining `none_instructions` after deduplication: {len(none_instructions)}")
else:
    print("Required lists (`none_instructions` or `valid_instructions`) are not available.")

Removed 0 items from `none_instructions` that were already processed.
Remaining `none_instructions` after deduplication: 6205


In [ ]:
filtered_none_instructions = []

for item in none_instructions:
    metadata = item.get('metadata', {})
    minecraft_version = metadata.get('minecraft_version')
    fabric_version = metadata.get('fabric_version')

    if minecraft_version is not None and fabric_version is not None:
        filtered_none_instructions.append(item)

print(f"Number of items with valid Minecraft and Fabric versions: {len(filtered_none_instructions)}")

Number of items with valid Minecraft and Fabric versions: 5625


In [ ]:
import gc
import torch

# 1. Shut down the executor — cancels any queued-but-not-started futures,
#    and waits for any actively running workers to finish (and hit their finally block)
if 'executor' in globals():
    executor.shutdown(wait=True, cancel_futures=True)
    del globals()['executor']

# 2. Drain model queue
if 'model_queue' in globals():
    while not model_queue.empty():
        m = model_queue.get()
        del m
    del globals()['model_queue']

# 3. Drain tokenizer queue
if 'tokenizer_queue' in globals():
    while not tokenizer_queue.empty():
        t = tokenizer_queue.get()
        del t
    del globals()['tokenizer_queue']

# 4. Delete any leftover references
for var in ['model', 'model_instance', 'tokenizer', 'tok']:
    if var in globals():
        del globals()[var]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Executor shut down, all models and tokenizers released.")

Executor shut down, all models and tokenizers released.


### Parallel Description Generation

This section automatically checks your available VRAM to determine how many 4-bit instances of the model can be loaded. It then uses a `ThreadPoolExecutor` and a `Queue` to run the generation in parallel, significantly speeding up the labeling process.

In [ ]:
import torch
import queue
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ── Configuration ────────────────────────────────────────────────────────────
BATCH_SIZE = 4                          # Tune down if you hit OOM; up for more throughput
MAX_CONTEXT_TOKENS = MAX_LABEL_TOKENS              # deepseek-coder-6.7b context window
MAX_NEW_TOKENS = 100
CACHE_CLEAR_INTERVAL = 10              # Clear CUDA cache every N batches
# ─────────────────────────────────────────────────────────────────────────────


# 1. VRAM Checker
def get_num_parallel_models():
    estimated_vram_gb = 6.5
    estimated_vram_bytes = estimated_vram_gb * 1024**3

    free_vram, total_vram = torch.cuda.mem_get_info()

    num_models = int(free_vram // estimated_vram_bytes)
    num_models = max(1, num_models)
    num_models = min(num_models, 4)

    print(f"Total VRAM: {total_vram / 1024**3:.2f} GB")
    print(f"Free VRAM:  {free_vram / 1024**3:.2f} GB")
    print(f"VRAM est. per instance: {estimated_vram_gb} GB")
    print(f"-> Loading {num_models} parallel model instance(s).")
    return num_models


num_models = get_num_parallel_models()


# 2. Load models and tokenizers into queues
#
# WHY a tokenizer queue instead of one shared tokenizer + a lock?
# The HuggingFace tokenizer is backed by a Rust object (tokenizers crate).
# Rust's borrow checker enforces that only one caller can hold a mutable
# reference at a time. Even with a Python threading.Lock, the Rust layer
# raises "Already borrowed" when two threads touch the same tokenizer
# instance concurrently. The only reliable fix is one tokenizer instance
# per worker thread — mirroring the model queue pattern.

model_queue = queue.Queue()
tokenizer_queue = queue.Queue()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"\nLoading {num_models} tokenizer and model instance(s) for {LABEL_MODEL}...")

for i in range(num_models):
    print(f"  [{i + 1}/{num_models}] Loading tokenizer...")
    tok = AutoTokenizer.from_pretrained(LABEL_MODEL, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    tok.padding_side = "left"   # Required for correct batch generation
    tokenizer_queue.put(tok)

    print(f"  [{i + 1}/{num_models}] Loading model...")
    model_instance = AutoModelForCausalLM.from_pretrained(
        LABEL_MODEL,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model_instance.eval()
    model_queue.put(model_instance)

print("All instances loaded.\n")


# 3. Token-aware prompt builder
# Accepts the calling thread's own tokenizer so no instance is ever shared
# across threads — even indirectly via a global.
def build_prompt(item: dict, tokenizer) -> str:
    """
    Builds a prompt, truncating the code section to fit within MAX_CONTEXT_TOKENS
    while leaving headroom for MAX_NEW_TOKENS.
    """
    file_type = item["metadata"].get("file_type", "unknown")
    prefix = (
        f"This is a Fabric Minecraft mod Java file of type '{file_type}'.\n"
        f"Write a single concise instruction (1-2 sentences) a developer might give "
        f"to produce this code.\n"
        f"Only output the instruction, nothing else.\n\n"
        f"Code:\n"
    )

    prefix_token_count = len(tokenizer.encode(prefix, add_special_tokens=False))
    max_code_tokens = MAX_CONTEXT_TOKENS - prefix_token_count - MAX_NEW_TOKENS

    code_token_ids = tokenizer.encode(
        item["code"],
        add_special_tokens=False,
        max_length=max_code_tokens,
        truncation=True,
    )
    truncated_code = tokenizer.decode(code_token_ids, skip_special_tokens=True)
    return prefix + truncated_code


# 4. Batch processing — each worker gets its own model + tokenizer from queues
def process_batch(batch: list[dict]) -> list[tuple[bool, dict]]:
    model = model_queue.get()
    tokenizer = tokenizer_queue.get()
    try:
        prompts = [build_prompt(item, tokenizer) for item in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_CONTEXT_TOKENS,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_k=50,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Use per-sequence attention mask to find each prompt's true end position.
        # With left-padding, sequences have different amounts of padding prepended,
        # so a single input_len would slice the wrong tokens for shorter prompts.
        attention_mask = inputs.attention_mask
        results = []
        for i, item in enumerate(batch):
            actual_input_len = attention_mask[i].sum().item()
            decoded = tokenizer.decode(
                outputs[i][actual_input_len:], skip_special_tokens=True
            ).strip()
            if decoded and "Error" not in decoded:
                item["instruction"] = decoded
                results.append((True, item))
            else:
                results.append((False, item))
        return results

    except Exception as e:
        print(f"[ERROR] Batch failed: {e}")
        return [(False, item) for item in batch]
    finally:
        # Always return both instances to their queues
        model_queue.put(model)
        tokenizer_queue.put(tokenizer)


# 5. Helper: split a list into fixed-size chunks
def chunked(lst: list, n: int):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


# 6. Main execution
if "filtered_none_instructions" in globals() and len(filtered_none_instructions) > 0:
    valid_instructions = globals().get("valid_instructions", [])
    batches = list(chunked(filtered_none_instructions, BATCH_SIZE))

    print(
        f"Starting parallel generation: {num_models} worker(s), "
        f"{len(filtered_none_instructions)} items, "
        f"{len(batches)} batches (batch size {BATCH_SIZE})..."
    )

    with ThreadPoolExecutor(max_workers=num_models) as executor:
        futures = {executor.submit(process_batch, batch): i for i, batch in enumerate(batches)}

        for completed_count, future in enumerate(
            tqdm(as_completed(futures), total=len(futures), desc="Generating")
        ):
            batch_results = future.result()
            for success, updated_item in batch_results:
                if success:
                    valid_instructions.append(updated_item)

            # Periodic VRAM cleanup to prevent fragmentation over long runs
            if completed_count % CACHE_CLEAR_INTERVAL == 0:
                torch.cuda.empty_cache()
                gc.collect()

    # Final cleanup
    torch.cuda.empty_cache()
    gc.collect()

    successful_shas = {item["metadata"]["sha"] for item in valid_instructions}
    none_instructions = [
        item for item in none_instructions
        if item["metadata"]["sha"] not in successful_shas
    ]

    print(f"\nDone!")
    print(f"  Valid instructions:     {len(valid_instructions)}")
    print(f"  Remaining (no result):  {len(none_instructions)}")

else:
    print("No filtered_none_instructions found. Run the separation cells above first.")

Total VRAM: 22.03 GB
Free VRAM:  21.84 GB
VRAM est. per instance: 6.5 GB
-> Loading 3 parallel model instance(s).

Loading 3 tokenizer and model instance(s) for Qwen/Qwen2.5-Coder-7B-Instruct...
  [1/3] Loading tokenizer...
  [1/3] Loading model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

  [3/3] Loading tokenizer...
  [3/3] Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

All instances loaded.

Starting parallel generation: 3 worker(s), 5625 items, 1407 batches (batch size 4)...


Generating:   0%|          | 0/1407 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Save and Upload Dataset

In [ ]:
#Deduplicate Again
if 'none_instructions' in globals() and 'valid_instructions' in globals():
    valid_shas = {item['metadata']['sha'] for item in valid_instructions}
    initial_none_count = len(none_instructions)

    # Filter out items from none_instructions that are already in valid_instructions
    # (identified by their SHA)
    none_instructions = [item for item in none_instructions if item['metadata']['sha'] not in valid_shas]

    deduplicated_count = initial_none_count - len(none_instructions)
    print(f"Removed {deduplicated_count} items from `none_instructions` that were already processed.")
    print(f"Remaining `none_instructions` after deduplication: {len(none_instructions)}")
else:
    print("Required lists (`none_instructions` or `valid_instructions`) are not available.")

Removed 2605 items from `none_instructions` that were already processed.
Remaining `none_instructions` after deduplication: 3600


In [ ]:
import json
from pathlib import Path

# Combine valid and none instructions
valid_instructions = globals().get('valid_instructions', [])
none_instructions = globals().get('none_instructions', [])

final_combined_dataset = valid_instructions + none_instructions

print(f"Total valid instructions: {len(valid_instructions)}")
print(f"Total none instructions: {len(none_instructions)}")
print(f"Total combined dataset size: {len(final_combined_dataset)}\n")

# Save the current session's work to a local JSONL file
with open("fabric_raw.jsonl", "w") as f:
    for example in final_combined_dataset:
        f.write(json.dumps(example) + "\n")

# Explore dataset metadata
if Path("fabric_raw.jsonl").exists() and len(final_combined_dataset) > 0:
    print("--- Updated Dataset Summary ---")
    print_dataset_statistics(final_combined_dataset)
else:
    print("No examples found to save yet.")

Total valid instructions: 2850
Total none instructions: 3600
Total combined dataset size: 6450

--- Updated Dataset Summary ---
--- Dataset Summary ---

Category Distribution (File Type):
file_type
general         4562
mixin           1170
player_logic     308
registry         220
block_item        81
entrypoint        72
datagen           37
Name: count, dtype: int64

Fabric Version Distribution:
fabric_version
0.16.0             694
0.139.4+1.21.11    662
None               580
0.18.4             498
0.114.0+1.21.4     311
                  ... 
0.138.0+1.21.10      5
0.134.0+1.21.9       4
0.97.2+1.20.4        4
0.121.0+1.21.5       4
0.102.0+1.21.1       3
Name: count, Length: 67, dtype: int64

Minecraft Version Distribution:
minecraft_version
1.21.11                                    1255
1.21                                        753
None                                        580
1.21.4                                      562
1.20.1                                      426
26

In [ ]:
from datasets import Dataset
import pandas as pd

if 'final_combined_dataset' in globals() and len(final_combined_dataset) > 0:
    # Convert the list of dictionaries to a HF Dataset via pandas to handle metadata
    df_final = pd.DataFrame(final_combined_dataset)
    full_dataset = Dataset.from_pandas(df_final)

    # Remove the pandas index column if it was added
    if "__index_level_0__" in full_dataset.column_names:
        full_dataset = full_dataset.remove_columns(["__index_level_0__"])

    # Shuffle the data and take a look at the features
    full_dataset = full_dataset.shuffle(seed=42)
    print(full_dataset)

    if 'instruction' in full_dataset.column_names:
        sample_inst = full_dataset[0].get('instruction')
        print(f"\nSample instruction:\n{sample_inst if sample_inst else 'None'}")
else:
    print("final_combined_dataset is empty or not defined.")

Dataset({
    features: ['instruction', 'context', 'code', 'metadata'],
    num_rows: 6450
})

Sample instruction:
This is a Fabric Minecraft mod Java file of type 'registry'.
Write a single concise instruction (1-2 sentences) a developer might give to produce this code.
Only output the instruction, nothing else.

Code:
package hunternif.mc.impl.atlas.network.packet.c2s.play;

import dev.architectury.networking.NetworkManager;
import hunternif.mc.impl.atlas.AntiqueAtlasMod;
import hunternif.mc.api.AtlasAPI;
import hunternif.mc.impl.atlas.network.packet.c2s.C2SPacket;
import hunternif.mc.impl.atlas.util.Log;
import net.minecraft.network.PacketByteBuf;
import net.minecraft.util.Identifier;
import net.minecraft.util.registry.Registry;
import net.minecraft.util.registry.RegistryKey;
import net.minecraft.world.World;

/**
 * Packet used to save the last browsing position for a dimension in an atlas.
 * @author Hunternif
 * @author Haven King
 */
public class PutBrowsingPositionC2SPacket ext

In [ ]:
import datetime

successful_upload = False
try:
    if 'full_dataset' in globals() and len(full_dataset) > 0:
        token = get_hf_token()
        if not token:
            raise ValueError("HF_TOKEN not found. Set HF_TOKEN in .env or Colab Secrets.")

        # Generate a commit message with current time and date
        current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        commit_message = f"Update dataset on {current_time} (Valid + None combined)"

        full_dataset.push_to_hub(hf_dataset_name, token=token, private=True, commit_message=commit_message)
        print(f"Successfully pushed {len(full_dataset)} examples to: https://huggingface.co/datasets/{hf_dataset_name}")
        successful_upload = True
    else:
        print("No examples found in 'full_dataset' to push.")
except Exception as e:
    print(f"Error pushing to Hub: {e}\nSet HF_TOKEN in .env or Colab Secrets and ensure write access to {hf_dataset_name}.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   7%|7         |  529kB / 7.32MB            

README.md:   0%|          | 0.00/606 [00:00<?, ?B/s]

Successfully pushed 6450 examples to: https://huggingface.co/datasets/marcuswu5/fabric-code-dataset-v2


In [ ]:
from google.colab import runtime
if AUTO_DISCONNECT and successful_upload:
  runtime.unassign()
  print("disconnected")


disconnected


# Finetune the Model

In [ ]:
from transformers import BitsAndBytesConfig

model_id = BASE_MODEL

# Configure 4-bit quantization formally
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the model for 4-bit training (vital for stable QLoRA)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Added MLP layers for better performance
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 39,976,960 || all params: 6,780,489,728 || trainable%: 0.5896


In [ ]:
# Define maximum sequence length
MAX_TOKENS = 15000

# Filter the dataset to limit token output/length
def filter_long_examples(example):
    # Approximate token count by characters or use tokenizer if loaded
    # Using a safe character-to-token heuristic (roughly 4 chars per token)
    return len(example['text']) < (MAX_TOKENS * 3.5)

filtered_dataset = full_dataset.filter(filter_long_examples)

# Perform a 90/10 random split
dataset_split = filtered_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
eval_dataset = dataset_split['test']

print(f"Original size: {len(full_dataset)}")
print(f"Filtered size: {len(filtered_dataset)}")
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(eval_dataset)}")

Filter:   0%|          | 0/3039 [00:00<?, ? examples/s]

Original size: 3039
Filtered size: 3038
Training set size: 2734
Validation set size: 304


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir="./fabric-coder-results",
        dataset_text_field="text",
        max_length=2048,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        optim="paged_adamw_8bit", # Changed from sgd to standard paged AdamW
        bf16=True,
        fp16=False,
        logging_steps=10,
        eval_steps=50,
        save_strategy="epoch",
        push_to_hub=False
    )
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Adding EOS to train dataset:   0%|          | 0/2734 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2734 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2734 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/304 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/304 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/304 [00:00<?, ? examples/s]

In [ ]:
%load_ext tensorboard
import matplotlib.pyplot as plt

# Start training
train_result = trainer.train()

# Extract log history
history = trainer.state.log_history

# Plotting
train_loss = [x['loss'] for x in history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
steps = [x['step'] for x in history if 'loss' in x]
eval_steps = [x['step'] for x in history if 'eval_loss' in x]

plt.figure(figsize=(10, 6))
plt.plot(steps, train_loss, label='Training Loss')
if eval_loss:
    plt.plot(eval_steps, eval_loss, label='Evaluation Loss')
plt.title('Training and Evaluation Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


Step,Training Loss
10,0.994426
20,0.834199


KeyboardInterrupt: 

In [ ]:
# Make sure to change "your-username" to your actual Hugging Face username!
repo_id = "marcuswu5/fabric-coder-adapter"

try:
    token = get_hf_token()
    if not token:
        raise ValueError("HF_TOKEN not found. Set HF_TOKEN in .env or Colab Secrets.")

    print(f"Saving final model locally...")
    trainer.save_model("./fabric-coder-final")

    print(f"Uploading LoRA adapters to {repo_id}...")
    # Push the model to Hugging Face
    trainer.model.push_to_hub(repo_id, token=token, private=True)

    print("Model successfully uploaded to Hugging Face!")
except Exception as e:
    print(f"Error uploading model: {e}\nSet HF_TOKEN in .env or Colab Secrets (write access required).")

# Dataset Filtering

## Data Selection Strategy
Data-efficient LLM Fine-tuning for Code Generation
Implementation of the data selection strategy from:
  "Data-efficient LLM Fine-tuning for Code Generation" (arXiv:2504.12687)

Strategy overview:
  1. Embed instructions via a sentence transformer
  2. K-Means cluster to preserve distributional consistency
  3. Score each sample with the IFD metric (instruction-following difficulty)
  4. Sample the top-m% highest-IFD samples from every cluster

Dependencies:
    pip install datasets sentence-transformers scikit-learn torch transformers tqdm

### Helpers

In [ ]:


from __future__ import annotations

import math
import logging
from typing import Optional

import torch
import numpy as np
from tqdm import tqdm

from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from transformers import AutoModelForCausalLM, AutoTokenizer

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# IFD scoring
# ---------------------------------------------------------------------------

# Cap NLL before exp() to prevent inf.  exp(700) ≈ 1e304, well within float64.
_MAX_NLL = 700.0


def _safe_exp(nll: float) -> float:
    """exp(nll) clamped to avoid overflow to inf."""
    return math.exp(min(nll, _MAX_NLL))


def _compute_perplexity_code_only(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    full_text: str,
    code_text: str,
    device: torch.device,
    max_length: int = 4096,
) -> float:
    """
    Return perplexity of the *code* portion of ``full_text``, with the
    non-code prefix tokens masked out of the loss (labels = -100).

    This is the correct way to compute PPL(C | I): the instruction tokens
    provide causal context but do not contribute to the loss.

    The code boundary is anchored to the **end** of the token sequence by
    tokenizing ``code_text`` separately (without special tokens) and using
    its length to determine how many trailing tokens belong to the response.
    This avoids off-by-one errors from BOS/EOS handling at the boundary.
    """
    # Tokenize the full sequence (with BOS etc.)
    full_ids: list[int] = tokenizer.encode(full_text)

    # Tokenize code alone WITHOUT special tokens so we get a clean token count.
    # We anchor from the end of the sequence so BOS boundaries don't matter.
    code_ids: list[int] = tokenizer.encode(code_text, add_special_tokens=False)
    n_code_tokens = len(code_ids)

    # Truncate full sequence to max_length (keep the tail so we preserve code)
    if len(full_ids) > max_length:
        full_ids = full_ids[-max_length:]

    n_full = len(full_ids)
    n_prefix = max(n_full - n_code_tokens, 0)

    # Build labels: -100 for prefix, real token ids for code
    labels = [-100] * n_prefix + full_ids[n_prefix:]

    input_ids = torch.tensor([full_ids], device=device)
    label_tensor = torch.tensor([labels], device=device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=label_tensor)

    # outputs.loss is mean NLL over non-masked (code) tokens
    return _safe_exp(outputs.loss.item())


def _compute_perplexity_text(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    text: str,
    device: torch.device,
    max_length: int = 4096,
) -> float:
    """Return perplexity of *text* (all tokens) under *model*."""
    input_ids = tokenizer.encode(text)
    if len(input_ids) > max_length:
        input_ids = input_ids[:max_length]

    id_tensor = torch.tensor([input_ids], device=device)

    with torch.no_grad():
        outputs = model(input_ids=id_tensor, labels=id_tensor)

    return _safe_exp(outputs.loss.item())


def _ifd_score(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    instruction: str,
    code: str,
    device: torch.device,
    instruction_template: str = "### Instruction:\n{instruction}\n\n### Response:\n{code}",
    code_only_template: str = "{code}",
    max_length: int = 4096,
) -> float:
    """
    Compute the Instruction Following Difficulty (IFD) score for one sample.

    IFD(C | I) = PPL(C | I) / PPL(C)

    ``PPL(C | I)`` is the perplexity of the *code tokens only* when the full
    instruction+code sequence is fed to the model (instruction tokens are
    masked from the loss).  ``PPL(C)`` is the perplexity of the code text
    alone.  A higher ratio means the code is harder to predict even given
    the instruction — i.e., the sample is more complex.

    Args:
        model: A causal LM loaded with AutoModelForCausalLM.
        tokenizer: Matching tokenizer.
        instruction: The natural-language instruction string.
        code: The corresponding code/response string.
        device: Torch device to run inference on.
        instruction_template: How to format the (instruction, code) pair.
            Must contain ``{instruction}`` and ``{code}`` placeholders.
        code_only_template: How to format code when no instruction is given.
            Must contain a ``{code}`` placeholder.
        max_length: Truncation length in tokens.

    Returns:
        IFD score (float ≥ 0). Lower means simpler; higher means more complex.
    """
    full_text = instruction_template.format(instruction=instruction, code=code)
    code_text = code_only_template.format(code=code)

    # Bug fix 1: PPL(C|I) must be computed on code tokens only.
    # Bug fix 2: _safe_exp prevents overflow to inf inside both helpers.
    ppl_with_instruction = _compute_perplexity_code_only(
        model, tokenizer, full_text, code_text, device, max_length
    )
    ppl_without_instruction = _compute_perplexity_text(
        model, tokenizer, code_text, device, max_length
    )

    # Guard against division-by-zero / nan propagation
    if ppl_without_instruction == 0 or not math.isfinite(ppl_without_instruction):
        return 0.0

    return ppl_with_instruction / ppl_without_instruction


def compute_ifd_scores(
    instructions: list[str],
    codes: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    device: torch.device,
    instruction_template: str = "### Instruction:\n{instruction}\n\n### Response:\n{code}",
    code_only_template: str = "{code}",
    max_length: int = 4096,
    batch_size: int = 1,        # increase if GPU memory allows
) -> list[float]:
    """
    Compute IFD scores for a list of (instruction, code) pairs.

    Args:
        instructions: List of instruction strings.
        codes: List of corresponding code/response strings.
        model: Causal LM for perplexity computation.
        tokenizer: Matching tokenizer.
        device: Torch device.
        instruction_template: See :func:`_ifd_score`.
        code_only_template: See :func:`_ifd_score`.
        max_length: Token truncation limit.
        batch_size: Currently processes one sample at a time (batch_size≥2
            requires custom loss masking; left as a future extension).

    Returns:
        List of IFD scores aligned with the input lists.
    """
    model.eval()
    scores: list[float] = []

    for instruction, code in tqdm(
        zip(instructions, codes),
        total=len(instructions),
        desc="Computing IFD scores",
    ):
        score = _ifd_score(
            model, tokenizer, instruction, code, device,
            instruction_template, code_only_template, max_length,
        )
        scores.append(score)

    return scores


# ---------------------------------------------------------------------------
# Embedding
# ---------------------------------------------------------------------------

def embed_instructions(
    instructions: list[str],
    embedding_model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    device: Optional[str] = None,
) -> np.ndarray:
    """
    Embed instructions using a SentenceTransformer model.

    Args:
        instructions: Raw instruction strings.
        embedding_model_name: Any model supported by sentence-transformers.
            The paper uses lightweight sentence transformers; the default
            ``all-MiniLM-L6-v2`` is a good balance of speed and quality.
        batch_size: Encoding batch size.
        device: ``"cuda"``, ``"cpu"``, etc.  Auto-detected when *None*.

    Returns:
        Float32 numpy array of shape ``(len(instructions), embedding_dim)``.
    """
    st_model = SentenceTransformer(embedding_model_name, device=device)
    embeddings = st_model.encode(
        instructions,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    return embeddings.astype(np.float32)


# ---------------------------------------------------------------------------
# Clustering + sampling
# ---------------------------------------------------------------------------

def cluster_and_sample(
    ifd_scores: list[float],
    cluster_labels: np.ndarray,
    n_clusters: int,
    sample_rate: float,
) -> list[int]:
    """
    Within each cluster, rank samples by IFD (descending) and return the
    indices of the top ``sample_rate`` fraction.

    Args:
        ifd_scores: IFD score for every sample in the dataset.
        cluster_labels: Cluster assignment for every sample (from K-Means).
        n_clusters: Total number of clusters *k*.
        sample_rate: Fraction of each cluster to keep (e.g. 0.4 → top 40 %).

    Returns:
        Sorted list of integer indices into the original dataset.
    """
    selected_indices: list[int] = []

    for cluster_id in range(n_clusters):
        # Indices of samples belonging to this cluster
        cluster_indices = np.where(cluster_labels == cluster_id)[0].tolist()

        if not cluster_indices:
            continue

        # Sort by IFD score descending (most complex first)
        cluster_indices.sort(key=lambda i: ifd_scores[i], reverse=True)

        # Take the top m%
        n_select = max(1, math.ceil(len(cluster_indices) * sample_rate))
        selected_indices.extend(cluster_indices[:n_select])

    return sorted(selected_indices)


### Public API



In [ ]:
def select_data(
    dataset: Dataset,
    model_name_or_path: str,
    *,
    instruction_column: str = "instruction",
    code_column: str = "output",
    sample_rate: float = 0.4,
    n_clusters: int = 10,
    embedding_model_name: str = "all-MiniLM-L6-v2",
    instruction_template: str = "### Instruction:\n{instruction}\n\n### Response:\n{code}",
    code_only_template: str = "{code}",
    max_token_length: int = 4096,
    embedding_batch_size: int = 64,
    kmeans_random_state: int = 42,
    device: Optional[str] = None,
    return_with_scores: bool = False,
) -> Dataset:
    """
    Select a high-quality subset of *dataset* using the strategy from:

        "Data-efficient LLM Fine-tuning for Code Generation" (arXiv:2504.12687)

    The three-step pipeline:

    1. **Embed** instructions with a lightweight sentence transformer.
    2. **Cluster** via K-Means to preserve the original data distribution.
    3. **Score** each sample with the IFD metric (ratio of perplexity with vs.
       without the instruction), then **sample** the top ``sample_rate``
       fraction from each cluster by IFD score.

    Args:
        dataset: A HuggingFace ``datasets.Dataset`` containing at minimum
            one instruction column and one code/response column.
        model_name_or_path: HuggingFace model ID or local path of the *same*
            base model you intend to fine-tune (used to compute IFD scores).
            The model must be a causal LM compatible with
            ``AutoModelForCausalLM``.
        instruction_column: Name of the column holding instruction strings.
        code_column: Name of the column holding code/response strings.
        sample_rate: Fraction of each cluster to retain.  The paper reports
            optimal results at 0.30–0.40.
        n_clusters: Number of K-Means clusters.  The paper uses 10.
        embedding_model_name: Sentence-transformer model for instruction
            embeddings.  Defaults to ``all-MiniLM-L6-v2`` (fast & compact).
        instruction_template: Template used when computing
            ``PPL(code | instruction)``.  Must contain ``{instruction}`` and
            ``{code}`` placeholders.
        code_only_template: Template used when computing ``PPL(code)``.
            Must contain a ``{code}`` placeholder.
        max_token_length: Maximum number of tokens for perplexity computation.
        embedding_batch_size: Batch size for sentence-transformer encoding.
        kmeans_random_state: Random seed for K-Means reproducibility.
        device: Torch device string.  Auto-detected when *None*.
        return_with_scores: If ``True``, the returned dataset will include two
            extra columns: ``"ifd_score"`` and ``"cluster_label"``.

    Returns:
        A new ``datasets.Dataset`` containing only the selected samples,
        optionally augmented with IFD scores and cluster labels.

    Example::

        from datasets import load_dataset
        from data_efficient_selection import select_data

        ds = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
        selected = select_data(
            ds,
            model_name_or_path="deepseek-ai/deepseek-coder-6.7b-base",
            instruction_column="problem",
            code_column="solution",
            sample_rate=0.4,
        )
        print(f"Selected {len(selected)} / {len(ds)} samples")
    """
    # ------------------------------------------------------------------
    # 0. Validate inputs
    # ------------------------------------------------------------------
    if instruction_column not in dataset.column_names:
        raise ValueError(
            f"Instruction column '{instruction_column}' not found. "
            f"Available columns: {dataset.column_names}"
        )
    if code_column not in dataset.column_names:
        raise ValueError(
            f"Code column '{code_column}' not found. "
            f"Available columns: {dataset.column_names}"
        )
    if not (0 < sample_rate <= 1):
        raise ValueError(f"sample_rate must be in (0, 1], got {sample_rate}")
    if n_clusters < 1:
        raise ValueError(f"n_clusters must be ≥ 1, got {n_clusters}")

    instructions: list[str] = dataset[instruction_column]
    codes: list[str] = dataset[code_column]
    n = len(dataset)
    logger.info("Dataset size: %d samples", n)

    # ------------------------------------------------------------------
    # 1. Embed instructions
    # ------------------------------------------------------------------
    logger.info("Step 1/3 — Embedding %d instructions …", n)
    embeddings = embed_instructions(
        instructions,
        embedding_model_name=embedding_model_name,
        batch_size=embedding_batch_size,
        device=device,
    )
    # Bug fix 3: explicitly free the sentence transformer from GPU memory
    # before loading the (much larger) causal LM.  Python's GC is
    # non-deterministic; without this, both models can sit on the GPU
    # simultaneously and cause OOM.
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # 2. K-Means clustering
    # ------------------------------------------------------------------
    effective_k = min(n_clusters, n)
    if effective_k < n_clusters:
        logger.warning(
            "n_clusters (%d) > dataset size (%d); reducing to %d.",
            n_clusters, n, effective_k,
        )

    logger.info("Step 2/3 — K-Means clustering (k=%d) …", effective_k)
    kmeans = KMeans(n_clusters=effective_k, random_state=kmeans_random_state, n_init="auto")
    cluster_labels: np.ndarray = kmeans.fit_predict(embeddings)

    # ------------------------------------------------------------------
    # 3. Load causal LM and compute IFD scores
    # ------------------------------------------------------------------
    if device is None:
        _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        _device = torch.device(device)

    logger.info("Step 3/3 — Loading model '%s' on %s …", model_name_or_path, _device)
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)
    lm = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        torch_dtype=torch.ffloat16 if _device.type == "cuda" else torch.float32,
        trust_remote_code=True,
    ).to(_device)

    logger.info("Computing IFD scores …")
    ifd_scores = compute_ifd_scores(
        instructions,
        codes,
        model=lm,
        tokenizer=tokenizer,
        device=_device,
        instruction_template=instruction_template,
        code_only_template=code_only_template,
        max_length=max_token_length,
    )

    # Free GPU memory — we no longer need the LM
    del lm
    if _device.type == "cuda":
        torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # 4. Sample top-m% from each cluster
    # ------------------------------------------------------------------
    logger.info(
        "Sampling top %.0f%% of each cluster …", sample_rate * 100
    )
    selected_indices = cluster_and_sample(
        ifd_scores, cluster_labels, effective_k, sample_rate
    )
    logger.info(
        "Selected %d / %d samples (%.1f%%)",
        len(selected_indices), n, 100 * len(selected_indices) / n,
    )

    # ------------------------------------------------------------------
    # 5. Build the output dataset
    # ------------------------------------------------------------------
    selected_dataset = dataset.select(selected_indices)

    if return_with_scores:
        selected_dataset = selected_dataset.add_column(
            "ifd_score", [ifd_scores[i] for i in selected_indices]
        )
        selected_dataset = selected_dataset.add_column(
            "cluster_label", [int(cluster_labels[i]) for i in selected_indices]
        )

    return selected_dataset

### Command Line Runner

In [ ]:
# ---------------------------------------------------------------------------
# Convenience: run from the command line
# ---------------------------------------------------------------------------

# if __name__ == "__main__":
#     import argparse
#     import json

#     logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

#     parser = argparse.ArgumentParser(
#         description="Select high-quality code-instruction samples (arXiv:2504.12687)."
#     )
#     parser.add_argument("--dataset", required=True,
#                         help="HuggingFace dataset ID or local path")
#     parser.add_argument("--split", default="train",
#                         help="Dataset split to load (default: train)")
#     parser.add_argument("--model", required=True,
#                         help="Causal LM used to score IFD (same as fine-tune target)")
#     parser.add_argument("--output_dir", required=True,
#                         help="Directory to save the selected dataset")
#     parser.add_argument("--instruction_column", default="instruction")
#     parser.add_argument("--code_column", default="output")
#     parser.add_argument("--sample_rate", type=float, default=0.4)
#     parser.add_argument("--n_clusters", type=int, default=10)
#     parser.add_argument("--embedding_model", default="all-MiniLM-L6-v2")
#     parser.add_argument("--device", default=None)
#     parser.add_argument("--return_with_scores", action="store_true")

#     args = parser.parse_args()

#     from datasets import load_dataset as _load_dataset

#     ds = _load_dataset(args.dataset, split=args.split)
#     selected = select_data(
#         ds,
#         model_name_or_path=args.model,
#         instruction_column=args.instruction_column,
#         code_column=args.code_column,
#         sample_rate=args.sample_rate,
#         n_clusters=args.n_clusters,
#         embedding_model_name=args.embedding_model,
#         device=args.device,
#         return_with_scores=args.return_with_scores,
#     )

#     selected.save_to_disk(args.output_dir)
#     stats = {
#         "original_size": len(ds),
#         "selected_size": len(selected),
#         "sample_rate": args.sample_rate,
#         "n_clusters": args.n_clusters,
#     }
#     print(json.dumps(stats, indent=2))

## test_suite: Colab Java benchmark

Run **Setup** once per Colab runtime after you have this repo on the VM (clone or Drive). Then load your model as usual and run **Benchmark** — it uses `model` and `tokenizer` from earlier cells (device is taken from the model).

Set `REPO_ROOT` to the folder that contains `test_suite/` (default `/content/Fabric-Coder`).


In [ ]:
# test_suite: one-time runtime prep (Java + PyYAML + paths)
import subprocess, sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

REPO_ROOT = Path("/content/Fabric-Coder")  # change if you cloned elsewhere
TEST_SUITE = REPO_ROOT / "test_suite"
sys.path.insert(0, str(TEST_SUITE))

import colab_eval
colab_eval.install_colab_java()
colab_eval.prepare_runtime(TEST_SUITE)

assert TEST_SUITE.is_dir(), f"Missing {TEST_SUITE} — clone repo or upload Fabric-Coder"
print("TEST_SUITE =", TEST_SUITE)
print("java:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[:1])


In [ ]:
# test_suite: run JUnit checks on model completions (expects `model` and `tokenizer` from above)
import colab_eval
import torch

device = next(model.parameters()).device

# Optional: only some tasks
# results = colab_eval.run_benchmark_on_model(TEST_SUITE, model, tokenizer, device, pair_ids=["int_point", "food_values"])

results = colab_eval.run_benchmark_on_model(TEST_SUITE, model, tokenizer, device)
colab_eval.print_report(results)
results
